<h2>Import requirements

In [10]:
import requests
import pandas as pd

<h2>Import match ids

<h4>These come from my games in the last 365 days

In [22]:
df_matches = pd.read_csv('data/my_match_ids.csv')
matches = df_matches['matchId'].tolist()
print(len(matches))
matches[:4]

64


[5521949939, 5521922599, 5476829612, 5476798099]

<h2>Get match data

<h4>Configure info for Riot API

In [ ]:
api_key = "api_key_here"
region = "americas"
headers = {"X-Riot-Token": api_key}

<h4>Make API calls and create dataframe

In [33]:
# dictionary to map team colors
team_color_map = {100: "blue", 200: "red"}

data_rows = []

i = 0
for match_id in matches: 
    # get match info json
    full_match_id = f"NA1_{match_id}"
    url = f"https://{region}.api.riotgames.com/lol/match/v5/matches/{full_match_id}"
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        match_data = response.json()
        info = match_data['info']
        # map team stats into dictionary
        team_stats = {}
        for team in info['teams']:
            team_stats[team['teamId']] = {
                "baron": team['objectives']['baron']['kills'],
                "dragon": team['objectives']['dragon']['kills'],
                "riftHerald": team['objectives']['riftHerald']['kills'],
                "tower": team['objectives']['tower']['kills']
            }
        # loop through match participants
        for player in info['participants']:
            t_id = player['teamId']   
            row = {
                "match_matchId": full_match_id,
                "match_gameDuration": info['gameDuration'],
                # player stats
                "player_teamId": team_color_map[t_id],
                "player_teamPosition": player['teamPosition'],
                "player_win": player['win'],
                "player_kills": player['kills'],
                "player_deaths": player['deaths'],
                "player_assists": player['assists'],
                "player_goldEarned": player['goldEarned'],
                "player_visionScore": player['visionScore'],
                "player_damageDealtToTurrets": player['damageDealtToTurrets'],
                # team stats
                "team_baron_kills": team_stats[t_id]['baron'],
                "team_dragon_kills": team_stats[t_id]['dragon'],
                "team_riftHerald_kills": team_stats[t_id]['riftHerald'],
                "team_tower_kills": team_stats[t_id]['tower']
            }
            # append row of player stats
            data_rows.append(row)
    else:
        print(response.status_code)
    if i!= 0 and i%10==0:
        print(f"First {i} matches complete...")
    i += 1

lol_data_df = pd.DataFrame(data_rows)

First 10 matches complete...
First 20 matches complete...
First 30 matches complete...
First 40 matches complete...
First 50 matches complete...
First 60 matches complete...


In [34]:
# print info for the first game to verify columns
lol_data_df.head(10)

,match_matchId,match_gameDuration,player_teamId,player_teamPosition,player_win,player_kills,player_deaths,player_assists,player_goldEarned,player_visionScore,player_damageDealtToTurrets,team_baron_kills,team_dragon_kills,team_riftHerald_kills,team_tower_kills
0,NA1_5521949939,955,blue,TOP,True,1,1,1,5304,5,0,0,2,0,1
1,NA1_5521949939,955,blue,JUNGLE,True,5,3,4,6025,10,1223,0,2,0,1
2,NA1_5521949939,955,blue,MIDDLE,True,6,2,2,6193,11,5151,0,2,0,1
3,NA1_5521949939,955,blue,BOTTOM,True,4,3,3,6130,11,2220,0,2,0,1
4,NA1_5521949939,955,blue,UTILITY,True,0,0,8,4163,20,303,0,2,0,1
5,NA1_5521949939,955,red,TOP,False,0,2,1,5601,11,5523,0,0,0,1
6,NA1_5521949939,955,red,JUNGLE,False,5,2,1,7190,16,0,0,0,0,1
7,NA1_5521949939,955,red,MIDDLE,False,2,6,1,4848,1,4244,0,0,0,1
8,NA1_5521949939,955,red,BOTTOM,False,2,2,1,5417,1,0,0,0,0,1
9,NA1_5521949939,955,red,UTILITY,False,0,4,3,3827,15,0,0,0,0,1


<h2>Clean and aggregate data

In [35]:
# make new unique match-team ids to avoid needing to use both columns later
lol_data_df["match_teamId"] = lol_data_df["match_matchId"] + lol_data_df["player_teamId"]

<h4>Isolate position-specific data

In [36]:
top = lol_data_df[lol_data_df["player_teamPosition"] == "TOP"]
jungle = lol_data_df[lol_data_df["player_teamPosition"] == "JUNGLE"]
middle = lol_data_df[lol_data_df["player_teamPosition"] == "MIDDLE"]
bottom = lol_data_df[lol_data_df["player_teamPosition"] == "BOTTOM"]
support = lol_data_df[lol_data_df["player_teamPosition"] == "UTILITY"]

<h4>Aggregate team stats by match

In [48]:
matches_df = lol_data_df[["match_teamId"]]
matches_df = matches_df.drop_duplicates().reset_index().drop(columns=["index"])
matches_df.head(2)

,match_teamId
0,NA1_5521949939blue
1,NA1_5521949939red


In [49]:
# save match duration
row_per_team = lol_data_df.drop_duplicates(subset="match_teamId")
matches_df["match_duration"] = matches_df["match_teamId"].map(row_per_team.set_index("match_teamId")["match_gameDuration"])


# save each role's stats
roles = {"top": top, "jg": jungle, "mid": middle, "bot": bottom, "sup": support}
for role in roles:
    matches_df = matches_df.merge(roles[role][["match_teamId", "player_kills", "player_deaths", "player_assists", "player_goldEarned", "player_visionScore", "player_damageDealtToTurrets"]], on="match_teamId", how="left"
                              ).rename(columns={"player_kills": f"{role}_kills", "player_deaths": f"{role}_deaths", "player_assists": f"{role}_assists", "player_goldEarned": f"{role}_gold", "player_visionScore": f"{role}_vision", "player_damageDealtToTurrets": f"{role}_tower_damage"})

# save team stats
matches_df["team_baron_kills"] = matches_df["match_teamId"].map(row_per_team.set_index("match_teamId")["team_baron_kills"])
matches_df["team_dragon_kills"] = matches_df["match_teamId"].map(row_per_team.set_index("match_teamId")["team_dragon_kills"])
matches_df["team_riftHerald_kills"] = matches_df["match_teamId"].map(row_per_team.set_index("match_teamId")["team_riftHerald_kills"])
matches_df["team_tower_kills"] = matches_df["match_teamId"].map(row_per_team.set_index("match_teamId")["team_tower_kills"])

# add new, aggregate team stats
team_kills = lol_data_df.groupby("match_teamId")["player_kills"].sum()
team_deaths = lol_data_df.groupby("match_teamId")["player_deaths"].sum()
team_assists = lol_data_df.groupby("match_teamId")["player_assists"].sum()
team_gold = lol_data_df.groupby("match_teamId")["player_goldEarned"].sum()
team_vision = lol_data_df.groupby("match_teamId")["player_visionScore"].sum()
team_tower_damage = lol_data_df.groupby("match_teamId")["player_damageDealtToTurrets"].sum()
matches_df["team_kills"] = matches_df["match_teamId"].map(team_kills)
matches_df["team_deaths"] = matches_df["match_teamId"].map(team_deaths)
matches_df["team_assists"] = matches_df["match_teamId"].map(team_assists)
matches_df["team_gold"] = matches_df["match_teamId"].map(team_gold)
matches_df["team_vision"] = matches_df["match_teamId"].map(team_vision)
matches_df["team_tower_damage"] = matches_df["match_teamId"].map(team_tower_damage)

# save match outcome
matches_df["team_win"] = matches_df["match_teamId"].map(row_per_team.set_index("match_teamId")["player_win"])

<h4>Remove early forfeit matches

In [50]:
early_forfeit_matches_df = matches_df[matches_df["match_duration"] <= 900]
print("Number of early forfeit matches: ", early_forfeit_matches_df.shape[0])
early_forfeit_matches_df.head(2)

Number of early forfeit matches:  6


,match_teamId,match_duration,top_kills,top_deaths,top_assists,top_gold,top_vision,top_tower_damage,jg_kills,jg_deaths,...,team_dragon_kills,team_riftHerald_kills,team_tower_kills,team_kills,team_deaths,team_assists,team_gold,team_vision,team_tower_damage,team_win
44,NA1_5470090822blue,755,6,1,1,6497,8,6042,1,2,...,2,0,1,9,6,3,21869,45,10753,True
45,NA1_5470090822red,755,0,3,0,2314,0,0,2,5,...,0,0,0,6,9,3,17264,27,749,False


In [51]:
matches_df = matches_df[matches_df["match_duration"] > 900]
print("Number of teams in final dataset: ", matches_df.shape[0])
print("Number of matches in final dataset: ", matches_df.shape[0] // 2)

Number of teams in final dataset:  122
Number of matches in final dataset:  61


In [52]:
matches_df.head(2)

,match_teamId,match_duration,top_kills,top_deaths,top_assists,top_gold,top_vision,top_tower_damage,jg_kills,jg_deaths,...,team_dragon_kills,team_riftHerald_kills,team_tower_kills,team_kills,team_deaths,team_assists,team_gold,team_vision,team_tower_damage,team_win
0,NA1_5521949939blue,955,1,1,1,5304,5,0,5,3,...,2,0,1,16,9,18,27815,57,8897,True
1,NA1_5521949939red,955,0,2,1,5601,11,5523,5,2,...,0,0,1,9,16,7,26883,44,9767,False


<h4>Remove "team_tower_kills" feature

In [53]:
matches_df = matches_df.drop(columns=["team_tower_kills"])

<h2>Save final cleaned dataset

<h4>Convert numbers to integer type and booleans to 0 / 1

In [54]:
matches_df[list(matches_df.columns)[1:]] = matches_df[list(matches_df.columns)[1:]].apply(pd.to_numeric, errors='coerce').astype("Int64")
matches_df[list(matches_df.columns)[1:]] = matches_df[list(matches_df.columns)[1:]].fillna(0)

<h4>Export to csv

In [55]:
matches_df = matches_df.drop(columns=["match_teamId"])
matches_df.to_csv('data/my_lol_dataset.csv', index=False)

<h2>Save normalized dataset

<h4>Divide numeric stat columns by match duration

In [56]:
cols_to_fix = matches_df.columns[1:-10].append(matches_df.columns[-7:-1])
matches_df[cols_to_fix] = matches_df[cols_to_fix].div(matches_df['match_duration'] / 60, axis=0) # matches_df["match_duration"] / 60 gives us the number of minutes
matches_df.head(1)

,match_duration,top_kills,top_deaths,top_assists,top_gold,top_vision,top_tower_damage,jg_kills,jg_deaths,jg_assists,...,team_baron_kills,team_dragon_kills,team_riftHerald_kills,team_kills,team_deaths,team_assists,team_gold,team_vision,team_tower_damage,team_win
0,955,0.062827,0.062827,0.062827,333.235602,0.314136,0.0,0.314136,0.188482,0.251309,...,0,2,0,1.005236,0.565445,1.13089,1747.539267,3.581152,558.973822,1


<h4>Rename effected columns

In [57]:
matches_df = matches_df.rename(columns={c: c + '_per_min' for c in cols_to_fix})

In [58]:
matches_df = matches_df.drop(columns=["match_duration"]) # this column isn't important if other columns are normalized, right?
matches_df.to_csv('data/my_lol_dataset_NORMALIZED.csv', index=False)